[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ZeruiW/frontier-ai-courses/blob/main/C49_Encoder_Seq2Seq_Course/00_setup/00_environment_check.ipynb)

# 00 · 环境自检与「三种掩码」热身

本课全程 **纯 numpy / CPU**，不加载预训练权重、不联网。

这个 notebook 做三件事：① 确认环境；② 用三十行把 **encoder-only / decoder-only / encoder-decoder 的注意力掩码**都写出来，
并用**数值梯度**证明它们的信息流差异；③ 立下全课的三条纪律：**与理论对拍 / 与朴素参考对拍 / 与公开量级对拍**。

## 1 · 环境自检

In [ ]:
import sys, platform, math
print('Python', sys.version.split()[0], '|', platform.system())
import numpy as np; print('numpy', np.__version__)
try:
    import pandas as pd; print('pandas', pd.__version__, '(可选)')
except Exception:
    print('pandas 未安装（可选，不影响课程）')
np.set_printoptions(precision=3, suppress=True)
print('环境就绪 ✅  —— 本课不需要 GPU / 预训练权重 / 联网')

## 2 · 三种掩码：Transformer 三种形态的全部差异源头

- **encoder-only（BERT）**：全 1 掩码，每个位置看得到所有位置 → 双向表示
- **decoder-only（GPT）**：下三角掩码，位置 t 只看得到 ≤ t → 可自回归
- **encoder-decoder（T5）**：编码器双向 + 解码器因果 + **交叉注意力全可见**

先把三种掩码造出来，并验证它们的形状性质。

In [ ]:
def bidirectional_mask(n):
    '''encoder-only：全部可见。'''
    return np.ones((n, n), dtype=bool)

def causal_mask(n):
    '''decoder-only：下三角（含对角线）。'''
    return np.tril(np.ones((n, n), dtype=bool))

def cross_mask(n_tgt, n_src):
    '''cross-attention：解码器每个位置都能看到编码器的全部位置。'''
    return np.ones((n_tgt, n_src), dtype=bool)

def prefix_lm_mask(n, n_prefix):
    '''prefix-LM（UniLM/GLM 用）：前缀内部双向，后缀因果。encoder/decoder 的一种融合。'''
    m = np.tril(np.ones((n, n), dtype=bool))
    m[:, :n_prefix] = True          # 所有位置都能看到整个前缀
    return m

N = 5
for name, m in [('bidirectional', bidirectional_mask(N)),
                ('causal', causal_mask(N)),
                ('prefix-LM (前缀=2)', prefix_lm_mask(N, 2))]:
    print(f'{name}:')
    print(np.where(m, 1, 0), '\n')

assert bidirectional_mask(N).all(), '双向掩码应全可见'
assert causal_mask(N)[0, 1:].sum() == 0, '因果掩码下位置 0 看不到未来'
assert causal_mask(N).sum() == N * (N + 1) // 2, '下三角元素数 = n(n+1)/2'
assert prefix_lm_mask(N, 2)[0, 1] and not prefix_lm_mask(N, 2)[2, 3], 'prefix-LM：前缀双向、后缀因果'
print('✅ 三种掩码的形状性质正确')

## 3 · 数值证明：因果掩码下未来 token 的梯度为零

这不是「据说」——是可以直接测出来的。
对一个单层注意力，计算 `∂ output[t] / ∂ input[t+1]`：
**因果掩码下必须严格为 0，双向掩码下必须非 0。** 这就是 BERT 不能自回归的根本原因。

In [ ]:
def softmax(x, axis=-1):
    x = x - x.max(axis=axis, keepdims=True)
    e = np.exp(x)
    return e / e.sum(axis=axis, keepdims=True)

def attention(X, Wq, Wk, Wv, mask):
    '''单头自注意力。X:(n,d)  mask:(n,n) bool'''
    Q, K, V = X @ Wq, X @ Wk, X @ Wv
    scores = Q @ K.T / math.sqrt(Q.shape[-1])
    scores = np.where(mask, scores, -1e9)
    return softmax(scores) @ V

rng = np.random.default_rng(0)
n, d = 6, 8
X = rng.normal(size=(n, d))
Wq, Wk, Wv = (rng.normal(size=(d, d)) * 0.3 for _ in range(3))

def numerical_jacobian_norm(mask, t, s, eps=1e-5):
    '''数值估计 ||∂output[t] / ∂input[s]||。'''
    base = attention(X, Wq, Wk, Wv, mask)[t]
    total = 0.0
    for k in range(d):
        Xp = X.copy(); Xp[s, k] += eps
        total += np.abs(attention(Xp, Wq, Wk, Wv, mask)[t] - base).sum() / eps
    return total

bi, ca = bidirectional_mask(n), causal_mask(n)
t, future = 2, 4          # 位置 2 的输出，对位置 4（未来）的输入的敏感度
g_bi = numerical_jacobian_norm(bi, t, future)
g_ca = numerical_jacobian_norm(ca, t, future)
print(f'双向掩码: ||∂out[{t}]/∂in[{future}]|| = {g_bi:.4f}')
print(f'因果掩码: ||∂out[{t}]/∂in[{future}]|| = {g_ca:.2e}')

assert g_bi > 1e-2, '双向掩码下未来信息必须流进来'
assert g_ca < 1e-6, '因果掩码下未来信息必须严格不可见'
print('\n✅ 数值证明：**双向注意力会让位置 t 看到 t+1**。')
print('   所以 BERT 若用「预测下一个 token」训练，等于抄答案 —— 损失瞬间归零，什么也学不到。')
print('   这一条就决定了 encoder-only 必须换一种预训练目标（MLM，模块 01）。')

### 把「抄答案」构造出来

上面证明了信息会泄漏。更强的一步：**在双向掩码下，存在一组参数让「预测下一个 token」的损失精确为 0**——
只要让每个位置 t 把注意力全部放到 t+1、把值向量原样搬过来、再与词嵌入表做内积即可。

**因果掩码下这组参数根本不存在**（位置 t+1 被屏蔽掉了）。这不是「训练得好不好」的问题，是可达性的问题。

In [ ]:
V = 12                                   # 迷你词表
Emb = np.random.default_rng(1).normal(size=(V, d))
Emb = Emb / np.linalg.norm(Emb, axis=1, keepdims=True) * 3.0   # 让内积可分

def cheat_attention_matrix(n):
    '''把注意力全押在下一个位置：A[t, t+1] = 1（最后一行只能看自己）。'''
    A = np.zeros((n, n))
    for t in range(n - 1):
        A[t, t + 1] = 1.0
    A[n - 1, n - 1] = 1.0
    return A

def loss_with_attention(A, toks):
    '''给定注意力矩阵，用「搬运值向量 + 与词表内积」的最简读出，算下一个 token 的交叉熵。
       只统计位置 0..n-2 —— 最后一个位置没有「下一个 token」可看（窗口边界）。'''
    H = A @ Emb[toks[:A.shape[1]]]
    logits = H @ Emb.T                    # 与词嵌入表内积 = 最近邻检索
    P = softmax(logits)
    m = A.shape[0] - 1                     # 排除最后一个位置
    tgt = toks[1:m + 1]
    return float(-np.log(P[np.arange(m), tgt] + 1e-12).mean())

toks = np.random.default_rng(3).integers(0, V, size=n + 1)
A_cheat = cheat_attention_matrix(n)
loss_cheat = loss_with_attention(A_cheat, toks)
print('作弊注意力矩阵 A[t,t+1]=1:')
print(A_cheat.astype(int))
print(f'\n用它做「预测下一个 token」的损失: {loss_cheat:.6f}  (随机基线 ln(V)={math.log(V):.3f})')
print('（最后一个位置没有「下一个 token」可看，不计入——这是窗口边界，不是模型的问题）')
assert loss_cheat < 0.05, '双向掩码下存在损失≈0 的「作弊解」'

# 关键：这组注意力在两种掩码下是否**可达**？
bi_ok = bool((A_cheat[~bidirectional_mask(n)] == 0).all())
ca_ok = bool((A_cheat[~causal_mask(n)] == 0).all())
print(f'\n作弊注意力在 双向掩码 下可达? {bi_ok}')
print(f'作弊注意力在 因果掩码 下可达? {ca_ok}   ← A[t,t+1] 恰好落在被屏蔽的上三角')
assert bi_ok and not ca_ok, '这才是 BERT 不能用 CLM 目标的根本原因'
print('\n✅ 不是「训练得好不好」的问题，是**可达性**的问题：')
print('   双向掩码下最优解就是抄答案（损失可以精确为 0，模型什么也没学到）；')
print('   因果掩码把这条捷径从参数空间里物理删除了。')
print('   MLM 的全部设计动机：保留双向上下文，同时人为制造一个「答案不在输入里」的预测任务。')

## 4 · 参数量账：三种形态的规模从哪来

一个 Transformer 的参数量几乎全在两处：**嵌入表** 与 **每层的 attention + FFN**。
把公式写出来，后面每个模块都会用它算账。

In [ ]:
def transformer_params(vocab, d_model, n_layers, d_ff=None, tie_embed=True,
                       n_dec_layers=0, cross_attn=False):
    '''返回参数量（近似，忽略 LayerNorm/bias 等小项）。'''
    d_ff = d_ff or 4 * d_model
    emb = vocab * d_model
    per_enc = 4 * d_model * d_model + 2 * d_model * d_ff      # QKVO + FFN(两层)
    per_dec = per_enc + (4 * d_model * d_model if cross_attn else 0)
    total = emb + n_layers * per_enc + n_dec_layers * per_dec
    if not tie_embed:
        total += vocab * d_model                              # 独立的输出投影
    return total

configs = [
    ('BERT-base   (enc-only, 12L)',  dict(vocab=30522, d_model=768,  n_layers=12)),
    ('BERT-large  (enc-only, 24L)',  dict(vocab=30522, d_model=1024, n_layers=24)),
    ('T5-base (enc-dec, 12+12L)',    dict(vocab=32128, d_model=768,  n_layers=12,
                                          n_dec_layers=12, cross_attn=True)),
    ('GPT-2 small (dec-only, 12L)',  dict(vocab=50257, d_model=768,  n_layers=12)),
]
print(f"{'配置':<32s} {'参数量(M)':>10s} {'嵌入占比':>9s}")
for name, cfg in configs:
    p = transformer_params(**cfg)
    emb_share = cfg['vocab'] * cfg['d_model'] / p
    print(f'{name:<32s} {p/1e6:>10.1f} {emb_share:>9.1%}')

p_bert  = transformer_params(vocab=30522, d_model=768, n_layers=12)
p_t5    = transformer_params(vocab=32128, d_model=768, n_layers=12, n_dec_layers=12, cross_attn=True)
assert 100e6 < p_bert < 130e6, f'BERT-base 应在 110M 量级，算得 {p_bert/1e6:.0f}M'
assert p_t5 > 2 * p_bert, 'encoder-decoder 参数量应显著大于同深同宽的 encoder-only'
print(f'\n✅ 与公开数字对拍：BERT-base ≈ 110M ✓')
print('   注意嵌入表占 BERT-base 的 ~21% —— 这就是 ALBERT 要做嵌入分解的原因（模块 02）。')

## 5 · ✏️ 练习：实现 prefix-LM 掩码并验证其性质

`prefix_lm_mask(n, n_prefix)` 已在上面给出。现在实现它的**逆问题**：
`infer_mask_type(mask)` —— 给定一个 (n,n) 的 bool 掩码，判断它属于哪种形态。
返回 `'bidirectional'` / `'causal'` / `'prefix-lm'` / `'other'`。

判据：全 1 → bidirectional；严格等于下三角 → causal；
存在 `k` (0<k<n) 使得「前 k 列全 1 且其余部分为下三角」→ prefix-lm；否则 other。

In [ ]:
def infer_mask_type(mask):
    # TODO: 按上述判据返回四种字符串之一
    raise NotImplementedError

In [ ]:
# —— 练习自测 ——
assert infer_mask_type(bidirectional_mask(5)) == 'bidirectional'
assert infer_mask_type(causal_mask(5)) == 'causal'
assert infer_mask_type(prefix_lm_mask(6, 2)) == 'prefix-lm'
assert infer_mask_type(prefix_lm_mask(6, 3)) == 'prefix-lm'
weird = causal_mask(5).copy(); weird[0, 4] = True
assert infer_mask_type(weird) == 'other'
# 边界：n_prefix=0 退化为 causal，n_prefix=n 退化为 bidirectional
assert infer_mask_type(prefix_lm_mask(5, 0)) == 'causal'
assert infer_mask_type(prefix_lm_mask(5, 5)) == 'bidirectional'
print('✅ 练习通过：三种形态是同一个掩码族上的三个点，prefix-LM 是它们之间的连续插值')

---
### 📖 参考答案

In [ ]:
def infer_mask_type(mask):
    n = mask.shape[0]
    if mask.all():
        return 'bidirectional'
    tril = np.tril(np.ones((n, n), dtype=bool))
    if np.array_equal(mask, tril):
        return 'causal'
    for k in range(1, n):
        cand = tril.copy(); cand[:, :k] = True
        if np.array_equal(mask, cand):
            return 'prefix-lm'
    return 'other'

## 6 · 立三条纪律

本课每个机制都会：

1. **与理论对拍** —— 实现要满足可证明的性质（如上面的梯度为零、softmax 归一、MLM 只在被掩位置计损失）。
2. **与朴素参考对拍** —— beam search vs 穷举、维特比 vs 暴力枚举路径、优化实现 vs 定义式实现。
3. **与公开量级对拍** —— 参数量、FLOPs、样本效率的账要与论文报告在同一量级。

把第三条封装成一个小工具。

In [ ]:
def check_magnitude(name, computed, reported, tol=0.25):
    '''与公开报告的数字比对，允许 tol 的相对误差（量级对拍，不是精确复现）。'''
    rel = abs(computed - reported) / reported
    ok = rel <= tol
    print(f'{name:<34s} 算得 {computed:>10,.1f} | 报告 {reported:>10,.1f} | 偏差 {rel:>6.1%} {"✅" if ok else "❌"}')
    return ok

ok1 = check_magnitude('BERT-base 参数量 (M)',
                      transformer_params(vocab=30522, d_model=768, n_layers=12) / 1e6, 110)
ok2 = check_magnitude('BERT-large 参数量 (M)',
                      transformer_params(vocab=30522, d_model=1024, n_layers=24) / 1e6, 340)
ok3 = check_magnitude('T5-base 参数量 (M)',
                      transformer_params(vocab=32128, d_model=768, n_layers=12,
                                         n_dec_layers=12, cross_attn=True) / 1e6, 220)
assert ok1 and ok2 and ok3, '参数量账应与公开数字在同一量级'
print('\n✅ 三条纪律就位。')

✅ 检查全部通过即环境就绪、方法论到位。

**本课的契约**：你写的每个机制（双向注意力、MLM 目标、分类/标注/抽取头、span corruption、cross-attention、beam search）都会
① 满足可证明的**理论性质**，② 与**朴素参考**给出相同结果，③ 参数量/效率的账与**公开报告**同量级。

**接下来五个模块**：01 MLM 与双向编码器 → 02 预训练目标的改良 → 03 下游微调三范式 → 04 Encoder-Decoder → 05 今天还要不要 encoder。

下一站：**模块 01 · MLM 与双向编码器** —— 既然不能预测下一个 token，那就把中间挖空。